# Sample & Explore — Darija Corpus

Pulls a cached sample from both corpora (YouTube + djelfa.info) so we can read real examples — script mix, French/English contamination, Arabizi, apparent MSA vs. Darija — to plan Phase 1 of the language/dialect filter. Doesn't write anything back to the corpus; purely exploratory.

## 1. Setup

In [1]:
import json
import random
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent  # Notebooks/ -> Darija/
DATA_DIR = ROOT / "Data"
DATA_DIR.mkdir(exist_ok=True)

SOURCES = {
    "youtube": sorted((ROOT / "Youtube_scrap" / "data" / "processed").glob("batch_*.jsonl")),
    "djelfa_info": sorted((ROOT / "Mountada_djelfa_scrap" / "data" / "processed").glob("batch_*.jsonl")),
}

SAMPLE_SIZE_PER_SOURCE = 1000
RANDOM_SEED = 42

for name, files in SOURCES.items():
    print(f"{name}: {len(files)} batch file(s)")
    for f in files:
        print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

youtube: 3 batch file(s)
  batch_2026-08-04.jsonl  (101.4 MB)
  batch_2026-08-05.jsonl  (105.6 MB)
  batch_2026-08-06.jsonl  (253.1 MB)
djelfa_info: 1 batch file(s)
  batch_2026-08-06.jsonl  (163.2 MB)


## 2. Cached reservoir sampling

Streams each batch file once (they're too large to load fully — YouTube's alone is 450MB+ across 3 files) and reservoir-samples `SAMPLE_SIZE_PER_SOURCE` docs per source. Cached to `Data/sample_<source>.jsonl` — reruns after a kernel restart just load the cache instead of re-scanning everything.

In [2]:
def reservoir_sample_jsonl(files, k, seed):
    rng = random.Random(seed)
    reservoir = []
    n_seen = 0
    for path in files:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                n_seen += 1
                if len(reservoir) < k:
                    reservoir.append(line)
                else:
                    j = rng.randint(0, n_seen - 1)
                    if j < k:
                        reservoir[j] = line
    return reservoir, n_seen


def load_or_build_sample(name, files, k, seed):
    cache_path = DATA_DIR / f"sample_{name}.jsonl"
    if cache_path.exists():
        print(f"{name}: loading cached sample from {cache_path.name}")
        with cache_path.open("r", encoding="utf-8") as f:
            docs = [json.loads(line) for line in f if line.strip()]
        return pd.DataFrame(docs)

    print(f"{name}: no cache found, sampling {k} docs from {len(files)} file(s)...")
    lines, n_seen = reservoir_sample_jsonl(files, k, seed)
    print(f"{name}: sampled {len(lines)} of {n_seen} total docs")

    with cache_path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")
    print(f"{name}: cached to {cache_path.name}")

    return pd.DataFrame(json.loads(line) for line in lines)


samples = {
    name: load_or_build_sample(name, files, SAMPLE_SIZE_PER_SOURCE, RANDOM_SEED)
    for name, files in SOURCES.items()
}
for name, df in samples.items():
    df["source"] = name

combined = pd.concat(samples.values(), ignore_index=True)
combined.shape

youtube: no cache found, sampling 1000 docs from 3 file(s)...


youtube: sampled 1000 of 975955 total docs
youtube: cached to sample_youtube.jsonl
djelfa_info: no cache found, sampling 1000 docs from 1 file(s)...


djelfa_info: sampled 1000 of 98310 total docs
djelfa_info: cached to sample_djelfa_info.jsonl


(2000, 13)

## 3. Basic stats

Doc counts and char/token-length distributions per source, as a sanity check against each project's `data/logs/log.json`.

In [3]:
print(combined.groupby("source").size().rename("sampled_docs"))
print()
print(combined.groupby("source")[["char_count", "token_count"]].describe().T)

source
djelfa_info    1000
youtube        1000
Name: sampled_docs, dtype: int64

source              djelfa_info      youtube
char_count  count   1000.000000  1000.000000
            mean     611.703000    51.998000
            std     1643.251731    77.903600
            min        9.000000     2.000000
            25%       68.000000    19.000000
            50%      159.000000    31.000000
            75%      446.750000    56.250000
            max    26804.000000  1400.000000
token_count count   1000.000000  1000.000000
            mean     110.768000     9.529000
            std      294.978824    14.526534
            min        1.000000     1.000000
            25%       12.000000     3.000000
            50%       29.000000     6.000000
            75%       81.000000    10.000000
            max     5129.000000   259.000000


## 4. Script-type split (throwaway heuristic)

The schema's `script` field is still `null` — that stage isn't built yet. This is a quick Arabic-character-ratio bucketing for *this notebook only* (not written back anywhere), just to see the rough Arabic-script / Latin-script(Arabizi) / mixed proportions in real data.

In [4]:
import re

ARABIC_RE = re.compile(r"[؀-ۿݐ-ݿࢠ-ࣿ]")
LATIN_RE = re.compile(r"[A-Za-z]")


def script_bucket(text, mixed_band=(0.15, 0.85)):
    arabic_chars = len(ARABIC_RE.findall(text))
    latin_chars = len(LATIN_RE.findall(text))
    total = arabic_chars + latin_chars
    if total == 0:
        return "no_letters"
    arabic_ratio = arabic_chars / total
    if arabic_ratio >= mixed_band[1]:
        return "arabic_script"
    if arabic_ratio <= mixed_band[0]:
        return "latin_script"
    return "mixed"


combined["script_bucket"] = combined["text"].map(script_bucket)

print(
    combined.groupby(["source", "script_bucket"]).size().unstack(fill_value=0)
)
print()
print(
    (combined.groupby(["source", "script_bucket"]).size() / combined.groupby("source").size())
    .unstack(fill_value=0)
    .round(3)
)

script_bucket  arabic_script  latin_script  mixed  no_letters
source                                                       
djelfa_info              968             7     25           0
youtube                  759           152     59          30

script_bucket  arabic_script  latin_script  mixed  no_letters
source                                                       
djelfa_info            0.968         0.007  0.025        0.00
youtube                0.759         0.152  0.059        0.03


## 5. Read real examples

Prints random examples per (source, script bucket) so we can actually eyeball what's in there — genuine Darija vs. MSA vs. French/English vs. noise/spam. Re-run the cell to get a different random draw.

In [5]:
def show_examples(df, source, bucket, n=8):
    subset = df[(df["source"] == source) & (df["script_bucket"] == bucket)]
    print(f"=== {source} / {bucket} ({len(subset)} in sample) ===")
    if subset.empty:
        print("(none in this sample)")
        return
    for text in subset["text"].sample(min(n, len(subset))):
        preview = text[:200].replace("\n", " ")
        print(f"- {preview}")
    print()


for source in combined["source"].unique():
    for bucket in ["arabic_script", "latin_script", "mixed"]:
        show_examples(combined, source, bucket, n=8)

=== youtube / arabic_script (759 in sample) ===
- كاين ناس ما يحبوش يحكو كيفي
- بزاف خو ههههه
- تعلم المونتاج و تزيدلوش
- لحمي تشووك يا جدك
- السلام عليكم ورحمة الله تعالى وبركاته واسعدا الله صباحكم بكل شيء جميل وتقبل الله منا ومنكم صالح الاعمال امين يارب العالمين 🤲🇩🇿❤️💪شكرا اختي ام وليد بارك الله فيك 👍🏻
- 3 دقايق مقدمة سيريو
- السلام عليكم  حبيت نسقسي على منصة يوكان هل جيدة ام شوبفاي افضل 13:16
- هذا جن من ناس يحوس علا حي في كوزينة

=== youtube / latin_script (152 in sample) ===
- Rahi 12.ta3 lil mahla yrdk makdrtch nrkd yarbiiiiiiiii🤥🤥🤥🤥🥳🥸😱😱😓😩😫🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱🥱
- Had l vidéo bayna beli 5demtha 3endek bezaf besa7 ki sheft l 9adia ta3 p diddy bedelt rayek ou dertha 3lih,te7sabetlek ma mregnaleksh hhhhhhhh😂
- Raklita hhh mklh ttal3na mais dhrlî f ki thooot jiha T3 la carte lbut mnha tfhmi parti mn les info personnelle onca daro cité beh nas thez m3lmaaat l3bae
- thabli machallah 😍🫀💌
- Ayoub peter qowa kho
- waluigi legend
- 😂😂😂😂😂😂😂🔥💎Tooop
- Yakho dir had lhkaya li 9ritha flektab mes kml h

## 6. Lexicon hit-rate preview

Runs the small function-word list from `Project_context.md` (both its Arabizi form and, since that list is Latin-only, common Arabic-script spellings of the same words so Arabic-script docs get a signal too) against the sample. This is an early read on whether that lexicon is discriminative enough before deciding how much to expand it for Phase 1 — not the real heuristic.

In [6]:
# Seed lexicon from Project_context.md (Arabizi form) + common Arabic-script
# spellings of the same words, so the preview covers both script types.
# NOTE: this is a small, hand-picked seed list for a quick preview only —
# Phase 1 should pull from a proper open Maghrebi/Darija word list, not this.
DARIJA_LEXICON = {
    "wach": ["wach", "wech", "واش"],
    "kayen": ["kayen", "kayn", "كاين", "كاينة"],
    "bezzaf": ["bezzaf", "bezaf", "بزاف"],
    "raki": ["raki", "rak", "راك", "راكي"],
    "chkoun": ["chkoun", "chkon", "شكون"],
    "khoya": ["khoya", "khouya", "خويا"],
    "wallah": ["wallah", "wllh", "والله"],
    "hna": ["hna", "hnaya", "هنا"],
    "kima": ["kima", "kimma", "كيما"],
}
ALL_VARIANTS = [v for variants in DARIJA_LEXICON.values() for v in variants]


def lexicon_hits(text):
    lowered = text.lower()
    return sum(1 for v in ALL_VARIANTS if v in lowered)


combined["lexicon_hits"] = combined["text"].map(lexicon_hits)

print(combined.groupby("source")["lexicon_hits"].describe())
print()
zero_hit_rate = (combined.groupby("source")["lexicon_hits"].apply(lambda s: (s == 0).mean())).round(3)
print("share of sample with zero lexicon hits:")
print(zero_hit_rate)

              count   mean       std  min  25%  50%  75%  max
source                                                       
djelfa_info  1000.0  0.277  0.631403  0.0  0.0  0.0  0.0  5.0
youtube      1000.0  0.212  0.524720  0.0  0.0  0.0  0.0  4.0

share of sample with zero lexicon hits:
source
djelfa_info    0.790
youtube        0.831
Name: lexicon_hits, dtype: float64


In [7]:
print("--- highest lexicon-hit examples ---")
top = combined.sort_values("lexicon_hits", ascending=False).head(8)
for _, row in top.iterrows():
    print(f"[{row['source']}/{row['script_bucket']}, hits={row['lexicon_hits']}] {row['text'][:200]!r}")

print()
print("--- zero-hit examples (random draw) ---")
zero = combined[combined["lexicon_hits"] == 0]
for text in zero["text"].sample(min(8, len(zero))):
    print(f"- {text[:200]!r}")

--- highest lexicon-hit examples ---
[djelfa_info/arabic_script, hits=5] 'اقتباس:\nالمشاركة الأصلية كتبت بواسطة أخت الرجال\nالسلام عليكم ورحمة الله تعالى وبركاته\nوعليكم السلام ورحمة الله وبركاته ومغفرته\nأولا\nمرحبا بأخي الصغير مليووووووووووون مرحبا\nوتفضّل لترتاح\nو مليار مرحبا '
[djelfa_info/arabic_script, hits=4] 'المستغفره\nالجلفه ما نعلافش الاسماوات بصحنروحو للسونت كيما راح المدفع ونكملو تلقاي بلاصه كومرسيال  انا نحب نروحلها عالى القماش  هربو علينا بزاف راهو في وحد السوق يومي  سقسي عيله يوروهلك  انا منين ندخله'
[youtube/arabic_script, hits=4] '@isleeembvnks2k60 راك غالط خويا القرين نتاعك حاجة باينة يكون يعرف وين تسكن بصح مايعرفش الغيبيات ولا واش كاين فعقلك ولا فقلبك ربي سبحانه برك لي علابالو غير ربي علابالو واش كاين فالصدور وراني متأكد واش '
[djelfa_info/arabic_script, hits=4] 'اقتباس:\nالمشاركة الأصلية كتبت بواسطة boukadoumzinou\nكاينة صيدلية مشهورة في عنابة يجيو ليها من 48 ولاية متخصصة في بيع  الادوية تاع الخارج و الادوية النادرة روح ليها متأكد راح تلقى الدواء لي تحتاجو\nللاس'


## 7. Notes / scratch

(Observations while reading through the examples above — lexicon coverage gaps, how much looks like French/English-only, MSA-vs-Darija judgment calls, anything surprising.)

*(write notes here)*